In [14]:
print("Hello, world!")

Hello, world!


In [23]:
from flax import nnx
import jax
import jax.numpy as jnp
import numpy as np

In [16]:
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, rngs: nnx.Rngs):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # W, k, v and out projections
        self.w_q = nnx.Linear(in_features=d_model, out_features=d_model, rngs=rngs)
        self.w_k = nnx.Linear(in_features=d_model, out_features=d_model, rngs=rngs)
        self.w_v = nnx.Linear(in_features=d_model, out_features=d_model, rngs=rngs)
        self.w_o = nnx.Linear(in_features=d_model, out_features=d_model, rngs=rngs)

        def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
            b, t, _ = x.shape
            
            # Project and split heads
            q = self.w_q(x).reshape(b, t, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
            k = self.w_k(x).reshape(b, t, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
            v = self.w_v(x).reshape(b, t, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
            
            # Scaled dot-product attention
            scores = (q @ k.transpose(0, 1, 3, 2)) / jnp.sqrt(self.d_k)
            attn_weights = jax.nn.softmax(scores, axis=-1)
            
            # Concat heads and project output
            context = attn_weights @ v
            out = context.transpose(0, 2, 1, 3).reshape(b, t, self.d_model)
            
            return self.w_o(out)

In [17]:
class FeedForward(nnx.Module):
    def __init__(self, d_model: int, ffn_dim: int, rngs: nnx.Rngs):
        self.linear1 = nnx.Linear(in_features=d_model, out_features=ffn_dim, rngs=rngs)
        self.linear2 = nnx.Linear(in_features=ffn_dim, out_features=d_model, rngs=rngs)
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        return self.linear2(jax.nn.gelu(self.linear1(x)))

In [18]:
class TransformerEncoderLayer(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, ffn_dim: int, rngs: nnx.Rngs):
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, rngs=rngs)
        self.ffn = FeedForward(d_model=d_model, ffn_dim=ffn_dim, rngs=rngs)
        
        # Define the two LayerNorm modules
        self.ln1 = nnx.LayerNorm(num_features=d_model)
        self.ln2 = nnx.LayerNorm(num_features=d_model)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        x1 = x + self.mha(self.ln1(x))
        return x1 + self.ffn(self.ln2(x1))

In [19]:
def get_positional_encoding(seq_len: int, d_model: int) -> jnp.ndarray:
    pos = jnp.arange(seq_len)[:, None]
    inv_freq = 1.0 / (10000 ** (jnp.arange(0, d_model, 2) / d_model))[None, :]
    angles = pos @ inv_freq
    
    pe = jnp.zeros((seq_len, d_model))
    pe = pe.at[:, 0::2].set(jnp.sin(angles))
    pe = pe.at[:, 1::2].set(jnp.cos(angles))
    return pe

In [20]:
class TransformerEncoder(nnx.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_heads: int, ffn_dim: int, rngs: nnx.Rngs):
        self.embed = nnx.Embed(num_embeddings=vocab_size, features_dim=d_model, rngs=rngs)
        
        # Instantiate independent layers
        self.layers = nnx.List([
            TransformerEncoderLayer(d_model, num_heads, ffn_dim, rngs) for _ in range(num_layers)
        ])

        self.d_model = d_model


    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        x = self.embed(x) + get_positional_encoding(x.shape[1], self.d_model)
        for layer in self.layers:
            x = layer(x)
        return x

In [21]:
class TransformerClassifier(nnx.Module):
    def __init__(self, encoder: TransformerEncoder, num_classes: int = 3, rngs: nnx.Rngs = 42):
        self.encoder = encoder
        self.fc = nnx.Linear(in_features=encoder.d_model, out_features=num_classes, rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        # Get sequence representations: (batch, seq_len, d_model)
        h = self.encoder(x)
        
        # Mean pooling across the sequence dimension (axis 1): (batch, d_model)
        pooled = jnp.mean(h, axis=1)
        
        # Project to class logits: (batch, num_classes)
        return self.fc(pooled)

In [24]:
# Define our vocabulary
CHARS = " 0123456789+="
VOCAB_SIZE = len(CHARS)
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARS)}
MAX_LEN = 16

In [26]:
def get_batch(batch_size: int) -> jnp.ndarray:
    equations = []
    for _ in range(batch_size):
        a = np.random.randint(0, 1000)
        b = np.random.randint(0, 1000)
        expr = f"{a}+{b}={a+b}".ljust(MAX_LEN)
        
        # Convert characters to integer IDs
        tokens = [CHAR_TO_IDX[c] for c in expr]
        equations.append(tokens)
        
    return jnp.array(equations, dtype=jnp.int32)